# Lesson 6: Coffea

Putting the pieces together: a columnar analysis framework for HEP.

## How to follow along, and Quizlets

Open the notebook named `lecture-workbook`, **not these slides**. Whenever you see a quizlet, do the exercise in your workbook and then we will go over the answer together.

Slides titled **(Optional)** contain live code examples. They need the `coffea` package and read `../data/SMHiggsToZZTo4L.root`, the same file as in earlier lessons.

## The Scikit-HEP ecosystem

<center>
<img src="../img/scikit-hep-shells.png" width="62%" alt="Scikit-HEP packages arranged in concentric shells around Python, NumPy and matplotlib">
</center>

<small>Figure: H. Schreiner, J. Pivarski, E. Rodrigues, <i>Awkward Packaging: building Scikit-HEP</i>, SciPy 2022.</small>

## Reading the shells: general at the core, specific on the outside

* **Core** (grey): the general-purpose scientific Python stack — Python, NumPy, matplotlib, Numba, Dask, Jupyter. Not HEP-specific at all.
* **Data structures and histograms**: Awkward Array (ragged data), boost-histogram, hist and uhi (histograms), plus fitting back-ends like iminuit and zfit.
* **HEP I/O and physics objects**: uproot (ROOT files), Vector (Lorentz vectors), Particle and hepunits — and **coffea**, which sits here and pulls all of the above together.
* **Statistics and plotting**: pyhf, cabinetry, hepstats, mplhep.
* **Outer shell**: specialised tools — FastJet, DecayLanguage, pyhepmc, uproot-browser, ...

## Connection to other lessons

| Lesson | Tool | Role in an analysis |
|:--|:--|:--|
| 2 | NumPy | one operation on a whole column |
| 3 | Classes | Processors, analysis_tools (Weights, PackedSelection), etc are classes |
| 4 | Awkward Array, uproot, Vector, hist | ragged event data, ROOT I/O, four-vectors, histograms |

Real analysis builds on these elements, layering on-top experiment-specific pieces: corrections, event weights with systematics, cutflows, book-keeping of many datasets. It needs a way to scale processing up to the terabytes of data typical in an analysis, with efficient workflows (compute, RAM, disk, network, ...)

Gluing them together by hand, in every analysis, is where **coffea** comes in.

## What is coffea?

**Co**lumnar **O**bject **F**ramework **F**or **E**ffective **A**nalysis — [github.com/scikit-hep/coffea](https://github.com/scikit-hep/coffea)

Sidenote: The developers pronounce it identically to the drink, others sometimes emphasize the `a`


Two ways to think about it:

**1. A "one-stop shop" for HEP analysis needs.**
One `pip install coffea` and you have I/O (uproot), ragged arrays with physics behaviour (Awkward + Vector), histogramming (hist), scale-out (futures, Dask, ...), analysis book-keeping, and specifically for CMS, corrections and their systematics — designed to work together.

**2. An incubator.** Ideas are prototyped inside coffea, and once they are proven and have undergone real-world testing, they tend to get factorized out into standalone packages (and get significant improvements in design in the process):

| Started in coffea as ... | Now lives in ... |
|:--|:--|
| `coffea.hist` | [hist](https://github.com/scikit-hep/hist) (based on boost-histogram backend) |
| `coffea.nanoevents.methods.vector` | [vector](https://github.com/scikit-hep/vector) |
| NanoEvents' column "zipping" into records | [awkward-zipper](https://github.com/CoffeaTeam/awkward-zipper) (WIP)|
| `lookup_tools`, `jetmet_tools`, `btag_tools`, ... (CMS corrections) | next in line to be spun out |

## Anatomy of a coffea analysis

<center>

| | step | coffea piece |
|:-:|:--|:--|
| 📂 | many ROOT files, grouped into datasets | *fileset* |
| ⬇ | | |
| 🧩 | flat branches → collections of physics objects | **1. NanoEvents + schemas** |
| ⬇ | | |
| 🔬 | select, correct, weight, histogram — *one chunk* | your `process()` function, using **3. analysis tools** and **4. corrections** |
| ⬇ | | |
| 🚀 | run every chunk, everywhere; add the results | **2. processor + executors** |
| ⬇ | | |
| 📊 | one set of histograms per dataset | → mplhep, pyhf, Combine, ... |

</center>

## 1. NanoEvents: applying a schema

A ROOT file stores *flat* branches: `Muon_pt`, `Muon_eta`, `Muon_phi`, `nMuon`, ...

A **schema** knows the naming conventions of a file format and turns those branches into one structured Awkward array (see "Struct of Array" SoA concept):

* **collections**: `events.Muon` is a ragged array of Muon records with fields `pt`, `eta`, ...
* **behaviours**: muons *are* Lorentz vectors — `.mass`, `.energy`, `+`, `.delta_r(...)` work out of the box
* **cross-references**: indices such as `Electron_genPartIdx` become links — `events.Electron.matched_gen.pdgId`
* **documentation**: each branch's help string travels with the array

Schemas exist for many formats: 
* `BaseSchema` (no interpretation, just the branches)
* `NanoAODSchema` (CMS)
* `PFNanoAODSchema` (CMS)
* `ScoutingNanoAODSchema` (CMS)
* `PHYSLITESchema` (ATLAS),
* `DelphesSchema`
* `TreeMakerSchema` (CMS Custom Format)
* `EDM4HEPSchema`
* `FCCSchema` (future colliders),
* `PDUNESchema` (DUNE)

## Loading only what you use

`NanoEventsFactory.from_root(..., mode=...)` has "three" modes:

| mode | what happens | use it for |
|:--|:--|:--|
| `"eager"` | read every branch now | tiny files, debugging - no additional I/O latency once initialized, but doesn't scale well! |
| `"virtual"` | read a branch the first time it is *used* | the default for processors |
| `"dask"`  | build a lazy Dask graph | task-graph based scale-out (*DEPRECATION IN PROGRESS!*)|
| `"graphed"` | build a lazy graph | task-graph based scale-out, smart systematics, ... (*WIP DASK REPLACEMENT BUILT FOR HEP*)|

A NanoAOD file has ~1500 branches; a typical analysis touches a few dozen. Reading only those provides a tremendous speedup, particularly when accessing files over remote connections (not local on disk)

## NanoEvents in code

With just the `BaseSchema`, we get the flat branches back, just like `uproot`
We'll use `eager` mode here because the file is small (40MB) and even with heavy compression will fit in RAM (figure of merit: 6x compression for standard CMS NanoAOD)

In [ ]:
import awkward as ak
from coffea.nanoevents import NanoEventsFactory, BaseSchema, NanoAODSchema

flat = NanoEventsFactory.from_root(
    {"../data/SMHiggsToZZTo4L.root": "Events"},
    schemaclass=BaseSchema,
    mode="eager",
).events()
flat.fields

With `NanoAODSchema`, branches are grouped into collections. A schema is just a class (lesson 3), so we can subclass it to adapt it — our file is a slimmed NanoAOD with no jets or photons, so we switch off the warnings about their missing cross-references:

In [ ]:
class SlimNanoAODSchema(NanoAODSchema):
    warn_missing_crossrefs = False


access_log_eager = []
events = NanoEventsFactory.from_root(
    {"../data/SMHiggsToZZTo4L.root": "Events"},
    schemaclass=SlimNanoAODSchema,
    metadata={"dataset": "SMHiggsToZZTo4L"},
    mode="eager",
    access_log=access_log_eager,
).events()
events.fields

## Collections have physics behaviour

In [ ]:
events.Muon

`energy` is not in the file; it comes from the Lorentz-vector behaviour:

In [ ]:
events.Muon.energy

While trivial, neither are `px` nor `pz` (easily calcultaed by hand from `pt` and `eta` and `phi`)

In [ ]:
events.Muon.px, events.Muon.pz

If we want to select two-$\mu$ events, we can do so like this...

In [ ]:
two_mu = events.Muon[ak.num(events.Muon) >= 2]
(two_mu[:, 0] + two_mu[:, 1]).mass

Only the columns we actually needed were read from the file:

In [ ]:
sorted({accessed.branch for accessed in access_log_eager})

## Quizlet 6.1 — electrons too

Compute the invariant mass of the two leading electrons, in events with at least two electrons.
Then look at `access_log` again: which new columns were read?

In [ ]:
two_e = events.Electron[ak.num(events.Electron) >= 2]

In [ ]:
(two_e[:, 0] + two_e[:, 1]).mass

In [ ]:
sorted({accessed.branch for accessed in access_log_eager})

## Quizlet 6.2 - try virtual mode

Create a `virtual_events` array using that separate mode, with an `access_log_virtual`, and compare the two

In [ ]:
access_log_virtual = []
virtual_events = NanoEventsFactory.from_root(
    {"../data/SMHiggsToZZTo4L.root": "Events"},
    schemaclass=SlimNanoAODSchema,
    metadata={"dataset": "SMHiggsToZZTo4L"},
    mode="virtual",
    access_log=access_log_virtual,
).events()

In [ ]:
virtual_events.Muon.energy, virtual_events.Muon.px, virtual_events.Muon.pz

In [ ]:
virtual_two_mu = virtual_events.Muon[ak.num(virtual_events.Muon) >= 2]
virtual_two_e = virtual_events.Electron[ak.num(virtual_events.Electron) >= 2]
print((virtual_two_mu[:, 0] + virtual_two_mu[:, 1]).mass, (virtual_two_e[:, 0] + virtual_two_e[:, 1]).pz)

In [ ]:
sorted({accessed.branch for accessed in access_log_virtual})

## 2. The processor: write the analysis for one chunk

```python
from coffea import processor

class MyAnalysis(processor.ProcessorABC):
    def process(self, events):
        # events: one chunk of one dataset, already wrapped by NanoEvents
        ...
        return {"mass": histogram, "entries": len(events)}

    def postprocess(self, accumulator):
        return accumulator
```

* You write `process()` as if the whole dataset were one array.
* It must return an **accumulator**: anything that can be added with `+` — numbers, histograms, (nested) dicts of those.
* coffea splits the input into chunks, calls `process()` on each, and adds the outputs: **map-reduce**.

## 2. Scale-out: same processor, different executor

```python
run = processor.Runner(
    executor=processor.IterativeExecutor(),   # ← the only line that changes
    schema=SlimNanoAODSchema,
    chunksize=100_000,
)
output = run(fileset, processor_instance=MyAnalysis())
```

| executor | runs on |
|:--|:--|
| `IterativeExecutor` | one core, in this process: debugging |
| `FuturesExecutor(workers=8)` | all cores of your laptop / login node |
| `DaskExecutor(client=...)` | a Dask cluster: HTCondor, Slurm, Kubernetes, analysis facilities like coffea-casa |
| `ParslExecutor`, `TaskVineExecutor` | other workflow systems on batch / HPC |

Your physics code does not change when you go from a laptop to a thousand cores.

A *fileset* says which files belong to which dataset, with per-dataset metadata:

```python
fileset = {
    "DYJets": {"files": {"root://.../dy_1.root": "Events", "root://.../dy_2.root": "Events"},
               "metadata": {"is_mc": True, "xsec": 6077.2}},
    "SingleMuon_2018A": {"files": {...}, "metadata": {"is_mc": False}},
}
```

## A processor in code

In [ ]:
import hist
import matplotlib.pyplot as plt
from coffea import processor


class DileptonMass(processor.ProcessorABC):
    def process(self, events):
        dataset = events.metadata["dataset"]
        mass = (
            hist.Hist.new
            .StrCat(["ee", "mumu"], name="channel")
            .Reg(120, 0, 120, name="mass", label=r"$m_{\ell\ell}$ [GeV]")
            .Double()
        )
        for channel, leptons in [("ee", events.Electron), ("mumu", events.Muon)]:
            pairs = leptons[ak.num(leptons) >= 2]
            mass.fill(channel=channel, mass=(pairs[:, 0] + pairs[:, 1]).mass)
        return {dataset: {"entries": len(events), "mass": mass}}

    def postprocess(self, accumulator):
        return accumulator

In [ ]:
fileset = {
    "SMHiggsToZZTo4L": {
        "files": {"../data/SMHiggsToZZTo4L.root": "Events"},
        "metadata": {"is_mc": True},
    },
}

run = processor.Runner(
    executor=processor.IterativeExecutor(),
    schema=SlimNanoAODSchema,
    chunksize=50_000,
)
output = run(fileset, processor_instance=DileptonMass())
output

## Scaling out is a one-line change

In [ ]:
run = processor.Runner(
    executor=processor.FuturesExecutor(workers=4),
    schema=SlimNanoAODSchema,
    chunksize=50_000,
)
output = run(fileset, processor_instance=DileptonMass())

output["SMHiggsToZZTo4L"]["mass"].plot()
plt.legend();

On a Dask cluster (see lesson 5), only the executor changes again, with options to setup different Dask clusters (local, remote, pre-prepared ones at analysis facilities like `coffea-casa` or LPC `Elastic Analysis Facility`):

```python
from dask.distributed import Client

run = processor.Runner(
    executor=processor.DaskExecutor(client=Client("tcp://127.0.0.1:8786")),
    schema=SlimNanoAODSchema,
    chunksize=50_000,
)
```

## 3. Analysis tools: `Weights`

Every simulated event carries a product of weights, and each factor has its own systematic uncertainty.

```python
from coffea.analysis_tools import Weights

weights = Weights(len(events))
weights.add("genWeight", events.genWeight)
weights.add("pileup", pu_nominal, weightUp=pu_up, weightDown=pu_down)
weights.add("muonID", sf, weightUp=sf_up, weightDown=sf_down)

weights.weight()               # product of all nominal weights
weights.weight("muonIDUp")     # same, with only the muon ID factor varied up
weights.variations             # {"pileupUp", "pileupDown", "muonIDUp", "muonIDDown"}
```

Filling one histogram per entry in `weights.variations` gives you all weight systematics for free.

## 3. Analysis tools: `PackedSelection`

Store many boolean event masks compactly (one *bit* per cut), then combine them:

```python
from coffea.analysis_tools import PackedSelection

selection = PackedSelection()
selection.add("twoMuons", ak.num(muons) == 2)
selection.add("oppositeSign", ak.sum(muons.charge, axis=1) == 0)
selection.add("trigger", events.HLT.IsoMu24)

selection.all("twoMuons", "oppositeSign", "trigger")          # AND of these
selection.require(twoMuons=True, oppositeSign=False)          # a same-sign control region
```

And from the same object, the two tables every analysis needs:

* `selection.cutflow(*cuts)` — events surviving each cut, applied one after another
* `selection.nminusone(*cuts)` — events surviving all cuts *but one*, for each cut: the N-1 plot that shows whether a cut is worth having

Both can `.print()`, make yield histograms (`.yieldhist()`), and histogram any variable at every step (`.plot_vars(...)`).

##A $H \to ZZ \to 4\ell$ selection in code

In [ ]:
import numpy as np
from coffea.analysis_tools import PackedSelection, Weights

muons = events.Muon[
    (events.Muon.pt > 5) & (abs(events.Muon.eta) < 2.4) & (events.Muon.pfRelIso04_all < 0.4)
]
electrons = events.Electron[
    (events.Electron.pt > 7) & (abs(events.Electron.eta) < 2.5) & (events.Electron.pfRelIso03_all < 0.4)
]

selection = PackedSelection()
selection.add("fourLeptons", ak.num(muons) + ak.num(electrons) == 4)
selection.add("chargeNeutral", ak.sum(muons.charge, axis=1) + ak.sum(electrons.charge, axis=1) == 0)
selection.add("sameFlavourPairs", ak.num(muons) % 2 == 0)
selection.add("leadingPt20", ak.any(muons.pt > 20, axis=1) | ak.any(electrons.pt > 20, axis=1))

cuts = list(selection.names)
cuts

## Cutflow and N-1

In [ ]:
cutflow = selection.cutflow(*cuts)
cutflow.print()

In [ ]:
selection.nminusone(*cuts).print()

In [ ]:
onecut, cumulative, labels = cutflow.yieldhist()
cumulative.plot(yerr=False)
plt.xticks(np.arange(len(labels)) + 0.5, labels, rotation=30)
plt.yscale("log");

## Quizlet 6.3 — one more cut

Add a cut `"fourMuons"` (exactly four selected muons) to `selection`,
and print the cutflow with it as the last step. How many $4\mu$ candidates are left?

In [ ]:
selection.add("fourMuons", ak.num(muons) == 4)
selection.cutflow(*cuts, "fourMuons").print()

## Weights with systematics, into one histogram

A toy muon-ID scale factor of $0.98 \pm 0.01$ per muon:

In [ ]:
weights = Weights(len(events))
weights.add(
    "muonID",
    weight=ak.prod(ak.full_like(muons.pt, 0.98), axis=1),
    weightUp=ak.prod(ak.full_like(muons.pt, 0.99), axis=1),
    weightDown=ak.prod(ak.full_like(muons.pt, 0.97), axis=1),
)

passing = selection.all(*cuts)
m4l = (muons.sum(axis=1) + electrons.sum(axis=1)).mass[passing]

m4l_hist = hist.Hist.new.StrCat([], name="systematic", growth=True).Reg(36, 70, 180, name="m4l", label=r"$m_{4\ell}$ [GeV]").Weight()
for systematic in [None, *sorted(weights.variations)]:
    m4l_hist.fill(
        systematic=systematic or "nominal",
        m4l=m4l,
        weight=weights.weight(systematic)[passing],
    )

m4l_hist.plot(overlay="systematic")
plt.legend();

## 4. CMS correction tools

Corrections turn "what the detector measured" into "what we believe happened", and come with uncertainties:

| module | what it does |
|:--|:--|
| `coffea.lookup_tools` | read ROOT histograms and CMS text formats into callable lookup tables that accept ragged arrays |
| `coffea.jetmet_tools` | jet energy scale and resolution (JEC/JER) with all uncertainty sources; propagate to MET |
| `coffea.btag_tools` | b-tagging scale factors |
| `coffea.lumi_tools` | `LumiMask` (golden JSON), integrated luminosity |
| Compatibility: [correctionlib](https://cms-nanoaod.github.io/correctionlib/) | the experiment-agnostic JSON format developed within CMS for (almost) all scale factors; coffea evaluates it directly on ragged arrays, and it works on numpy arrays, and the format is portable with C++ headers (permitting it to be utilized in C++, whether with ROOT or without) |

These subtools are CMS-specific, so they are **planned to be spun out of coffea** into their own standalone package — the incubator pattern again.

In a real CMS analysis, the corrections come from the POG's published JSON files:

```python
import correctionlib
from coffea.lumi_tools import LumiMask

# Note: Golden JSONs are NOT correctionlib files, so specific lumi_tools parse them
events = events[LumiMask("Cert_Collisions2018_golden.json")(events.run, events.luminosityBlock)]

muon_sf = correctionlib.CorrectionSet.from_file("muon_Z.json.gz")["NUM_TightID_DEN_TrackerMuons"]
sf = muon_sf.evaluate(abs(muons.eta), muons.pt, "nominal")   # ragged in, ragged out
weights.add("muonID", ak.prod(sf, axis=1), ...)
```

## (Optional) A correction in code

Build a (toy) muon scale factor binned in $|\eta|$ and $p_T$ as a histogram, and convert it to correctionlib:

In [ ]:
import correctionlib
import correctionlib.convert

sf_hist = hist.Hist(
    hist.axis.StrCategory(["nominal", "up", "down"], name="variation"),
    hist.axis.Variable([0, 1.2, 2.4], name="abseta"),
    hist.axis.Variable([5, 20, 50, 1000], name="pt"),
    name="toy_muon_id",
    label="weight",
)
nominal = np.array([[0.95, 0.97, 0.98], [0.93, 0.96, 0.97]])
sf_hist.view()[...] = np.stack([nominal, nominal + 0.01, nominal - 0.01])

correction = correctionlib.convert.from_histogram(sf_hist, flow="clamp")
correction.description = "Toy muon ID scale factors"
correction.output.description = "The toy muon scale factor weight"
correction_set = correctionlib.schemav2.CorrectionSet(schema_version=2, corrections=[correction])
muon_sf = correction_set.to_evaluator()["toy_muon_id"]

In [ ]:
# CorrectionSets are self-documenting, moreso with more information attached... a good generalized lookup format
import rich
rich.print(correction_set)

## (Optional) Evaluate on ragged muons, and use it as a weight

In [ ]:
muon_sf.evaluate("nominal", abs(muons.eta), muons.pt)

In [ ]:
weights = Weights(len(events))
weights.add(
    "muonID",
    weight=ak.prod(muon_sf.evaluate("nominal", abs(muons.eta), muons.pt), axis=1),
    weightUp=ak.prod(muon_sf.evaluate("up", abs(muons.eta), muons.pt), axis=1),
    weightDown=ak.prod(muon_sf.evaluate("down", abs(muons.eta), muons.pt), axis=1),
)
weights.weight("muonIDUp")[passing] / weights.weight()[passing]

## Summary

* coffea sits on top of the Scikit-HEP stack: uproot, Awkward, Vector, hist, Dask, correctionlib.
* It is a **one-stop shop** for analysis, and an **incubator** whose successes become standalone packages.
* **NanoEvents** + **schemas**: flat branches → physics objects with behaviour, read lazily.
* **Processors** + **xecutors**: write `process()` for one chunk; run it anywhere by swapping the executor.
* **Analysis tools**: `Weights` for weights and their systematics; `PackedSelection` for regions, simple cutflows and N-1 plots.
* **CMS corrections**: lookup tables, JEC/JER, b-tagging, lumi masks, (with correctionlib compatibility) — soon a package of their own.

## Where to go next

* Documentation and tutorials: [coffea-hep.readthedocs.io](https://coffea-hep.readthedocs.io)
* Source, issues and discussions: [github.com/scikit-hep/coffea](https://github.com/scikit-hep/coffea)
* Longer hands-on training (NanoEvents, corrections, analysis tools, processors, scale-out): [github.com/iris-hep/uscms-idap-training](https://github.com/iris-hep/uscms-idap-training), `coffea/` directory
* A complete analysis, from files to a fit: the [Analysis Grand Challenge](https://agc.readthedocs.io)